In [1]:
!pip install polars

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 828.7/828.7 kB 65.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 204.2 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [polars]2m1/2 [polars]


In [2]:
import os
import polars as pl

In [3]:
os.chdir("/home/jovyan/")
WORK_DIR = os.getcwd()
print (WORK_DIR)

DATA = os.path.join(WORK_DIR, "data")
print(DATA)

/home/jovyan
/home/jovyan/data


In [4]:
# pl.scan_csv doesn't load the file into memory immediately but only when called with the
# .collect() method, which generally makes running the code significantly faster.
# schema_overrides is optional and can be used to explicitly set a data type to a column,
# but it will return an error if polars finds some kind of mismatch.
def get_data(file: str, separator: str = ",", schema_overrides: dict = None) -> pl.LazyFrame:
    return pl.scan_csv(source=f"{DATA}/{file}", 
                       separator=separator, 
                       schema_overrides=schema_overrides,
                      decimal_comma=True)

In [5]:
yield_curve_scheme = {
    "date": pl.Date,
    "yield_percentage": pl.Float64,
    "maturity_month": pl.Float64
}

equity_data_scheme = {
    "Date": pl.Date
}

# pl.Datetime("ns") is accurate to the nano-second.
quotes_inc_eu_schema = {
    "trade_id": pl.Int128,
    "event_timestamp": pl.Datetime("ns")
}

trades_eu_schema = {
    "trade_id": pl.Int128,
    "event_timestamp": pl.Datetime("ns")
}

quotes_inc_us_schema = {
    "trade_id": pl.Int128,
    "event_timestamp": pl.Datetime("ns")
}

trades_us_schema = {
    "trade_id": pl.Int128,
    "event_timestamp": pl.Datetime("ns")
}

In [6]:
yield_curve_file = "yield_curve.csv"
equity_data_file = "equity_data.csv"
quotes_inc_eu_file = "DE0007500001_quotes_incremental.csv"
trades_eu_file = "DE0007500001_trades.csv"
quotes_inc_us_file = "US2561631068_quotes_incremental.csv"
trades_us_file = "US2561631068_trades.csv"

In [7]:
# Yield Curve
# The original file was an .xlsx file, which I've modified to a .csv
# Also, I've slightly modified the files to a proper tabular format,
# adding the reference date (1989-03-31) as a column. The other values
# remain unchanged.
yield_curve = get_data(file = yield_curve_file, 
                       separator = ";", 
                       schema_overrides=yield_curve_scheme)
yield_curve = yield_curve.drop_nulls()

print(yield_curve.collect())
print(yield_curve.collect().columns)
print(yield_curve.collect().dtypes)
print(yield_curve.collect().describe())

shape: (17, 3)
┌────────────┬──────────────────┬────────────────┐
│ date       ┆ yield_percentage ┆ maturity_month │
│ ---        ┆ ---              ┆ ---            │
│ date       ┆ f64              ┆ f64            │
╞════════════╪══════════════════╪════════════════╡
│ 1989-03-31 ┆ 9.12             ┆ 3.0            │
│ 1989-03-31 ┆ 9.32             ┆ 6.0            │
│ 1989-03-31 ┆ 9.34             ┆ 9.0            │
│ 1989-03-31 ┆ 9.62             ┆ 12.0           │
│ 1989-03-31 ┆ 9.69             ┆ 15.0           │
│ …          ┆ …                ┆ …              │
│ 1989-03-31 ┆ 9.25             ┆ 72.0           │
│ 1989-03-31 ┆ 9.15             ┆ 84.0           │
│ 1989-03-31 ┆ 9.12             ┆ 96.0           │
│ 1989-03-31 ┆ 9.05             ┆ 108.0          │
│ 1989-03-31 ┆ 9.0              ┆ 120.0          │
└────────────┴──────────────────┴────────────────┘
['date', 'yield_percentage', 'maturity_month']
[Date, Float64, Float64]
shape: (9, 4)
┌────────────┬──────────────────

In [8]:
# The original file was an .xlsx file which I've converted to csv.
equity_data = get_data(file = equity_data_file, separator = ";", schema_overrides=equity_data_scheme)

print(equity_data.collect())
print(equity_data.collect().columns)
print(equity_data.collect().dtypes)
print(equity_data.collect().describe())

shape: (1_304, 41)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ Date      ┆ ADIDAS -  ┆ AIRBUS -  ┆ ALLIANZ - ┆ … ┆ SYMRISE - ┆ VOLKSWAGE ┆ VONOVIA - ┆ ZALANDO  │
│ ---       ┆ TOT       ┆ TOT       ┆ TOT       ┆   ┆ TOT       ┆ N PREF. - ┆ TOT       ┆ - TOT    │
│ date      ┆ RETURN    ┆ RETURN    ┆ RETURN    ┆   ┆ RETURN    ┆ TOT       ┆ RETURN    ┆ RETURN   │
│           ┆ IND       ┆ IND       ┆ IND       ┆   ┆ IND       ┆ RETURN …  ┆ IND       ┆ IND      │
│           ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│           ┆ f64       ┆ f64       ┆ f64       ┆   ┆ f64       ┆ f64       ┆ f64       ┆ f64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 2021-01-0 ┆ 4124.59   ┆ 760.55    ┆ 8424.11   ┆ … ┆ 835.99    ┆ 1655.24   ┆ 456.22    ┆ 423.63   │
│ 1         ┆           ┆           ┆           ┆   ┆           ┆       

In [9]:
## EU: Quotes Incremental
quotes_inc_eu = get_data(file=quotes_inc_eu_file, schema_overrides=quotes_inc_eu_schema)

print(quotes_inc_eu.collect().head())
print(quotes_inc_eu.collect().columns)
print(quotes_inc_eu.collect().dtypes)
print(quotes_inc_eu.collect().describe())

shape: (5, 34)
┌──────┬───────┬───────┬───────────────┬───┬───────────────┬───────────────┬───────────────┬───────┐
│ side ┆ price ┆ size  ┆ order_id      ┆ … ┆ total_ask_ord ┆ total_bid_ord ┆ market_state  ┆ venue │
│ ---  ┆ ---   ┆ ---   ┆ ---           ┆   ┆ ers           ┆ ers           ┆ ---           ┆ ---   │
│ str  ┆ str   ┆ i64   ┆ i64           ┆   ┆ ---           ┆ ---           ┆ str           ┆ str   │
│      ┆       ┆       ┆               ┆   ┆ i64           ┆ i64           ┆               ┆       │
╞══════╪═══════╪═══════╪═══════════════╪═══╪═══════════════╪═══════════════╪═══════════════╪═══════╡
│ BID  ┆ 7.074 ┆ 1828  ┆ 1081350376584 ┆ … ┆ 0             ┆ 1             ┆ CONTINUOUS_TR ┆ CEUX  │
│      ┆       ┆       ┆ 637819        ┆   ┆               ┆               ┆ ADING         ┆       │
│ ASK  ┆ 7.164 ┆ 1828  ┆ 1081350376584 ┆ … ┆ 1             ┆ 1             ┆ CONTINUOUS_TR ┆ CEUX  │
│      ┆       ┆       ┆ 637820        ┆   ┆               ┆               ┆

In [10]:
## EU: Trades
trades_eu = get_data(file=trades_eu_file, schema_overrides=trades_eu_schema)

print(trades_eu.collect().head())
print(trades_eu.collect().columns)
print(trades_eu.collect().dtypes)
print(trades_eu.collect().describe())

shape: (5, 9)
┌────────────┬────────────┬────────────┬───────────┬───┬───────────┬───────────┬───────────┬───────┐
│ trade_id   ┆ trade_time ┆ publicatio ┆ aggressor ┆ … ┆ execution ┆ market_st ┆ trade_typ ┆ venue │
│ ---        ┆ stamp      ┆ n_timestam ┆ _side     ┆   ┆ _size     ┆ ate       ┆ e         ┆ ---   │
│ i128       ┆ ---        ┆ p          ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ str   │
│            ┆ str        ┆ ---        ┆ str       ┆   ┆ i64       ┆ str       ┆ str       ┆       │
│            ┆            ┆ str        ┆           ┆   ┆           ┆           ┆           ┆       │
╞════════════╪════════════╪════════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════╡
│ 6269133135 ┆ 2023-09-01 ┆ 2023-09-01 ┆ ASK       ┆ … ┆ 166       ┆ CONTINUOU ┆ LIT       ┆ CEUX  │
│ 937        ┆ 07:00:23.8 ┆ 07:00:23.8 ┆           ┆   ┆           ┆ S_TRADING ┆           ┆       │
│            ┆ 34308000   ┆ 34308000   ┆           ┆   ┆           ┆         

In [13]:
## US: Quotes Incremental
quotes_inc_us = get_data(file=quotes_inc_us_file, schema_overrides=quotes_inc_us_schema)

print(quotes_inc_us.collect().head())
print(quotes_inc_us.collect().columns)
print(quotes_inc_us.collect().dtypes)
print(quotes_inc_us.collect().describe())

shape: (5, 37)
┌─────┬──────┬───────┬──────┬───┬──────────────────┬──────────────────┬──────────────┬───────┐
│     ┆ side ┆ price ┆ size ┆ … ┆ total_ask_orders ┆ total_bid_orders ┆ market_state ┆ venue │
│ --- ┆ ---  ┆ ---   ┆ ---  ┆   ┆ ---              ┆ ---              ┆ ---          ┆ ---   │
│ i64 ┆ str  ┆ str   ┆ i64  ┆   ┆ i64              ┆ i64              ┆ str          ┆ str   │
╞═════╪══════╪═══════╪══════╪═══╪══════════════════╪══════════════════╪══════════════╪═══════╡
│ 0   ┆ BID  ┆ 35.41 ┆ 1878 ┆ … ┆ 0                ┆ 1                ┆ PRE_OPEN     ┆ XASE  │
│ 1   ┆ BID  ┆ 47.25 ┆ 100  ┆ … ┆ 0                ┆ 2                ┆ PRE_OPEN     ┆ XASE  │
│ 2   ┆ ASK  ┆ 54.18 ┆ 100  ┆ … ┆ 1                ┆ 2                ┆ PRE_OPEN     ┆ XASE  │
│ 3   ┆ BID  ┆ 50.16 ┆ 1500 ┆ … ┆ 1                ┆ 3                ┆ PRE_OPEN     ┆ XASE  │
│ 4   ┆ ASK  ┆ 50.9  ┆ 1500 ┆ … ┆ 2                ┆ 3                ┆ PRE_OPEN     ┆ XASE  │
└─────┴──────┴───────┴──────┴───┴──

In [12]:
## US: Trades
trades_us = get_data(file=trades_us_file, schema_overrides=trades_us_schema)

print(trades_us.collect().head())
print(trades_us.collect().columns)
print(trades_us.collect().dtypes)
print(trades_us.collect().describe())

shape: (5, 15)
┌─────┬──────────┬─────────────┬─────────────┬───┬─────────────┬─────────────┬─────────────┬───────┐
│     ┆ trade_id ┆ trade_times ┆ publication ┆ … ┆ bmll_trade_ ┆ trade_actio ┆ execution_v ┆ venue │
│ --- ┆ ---      ┆ tamp        ┆ _timestamp  ┆   ┆ type        ┆ n           ┆ enue        ┆ ---   │
│ i64 ┆ i128     ┆ ---         ┆ ---         ┆   ┆ ---         ┆ ---         ┆ ---         ┆ str   │
│     ┆          ┆ str         ┆ str         ┆   ┆ str         ┆ str         ┆ str         ┆       │
╞═════╪══════════╪═════════════╪═════════════╪═══╪═════════════╪═════════════╪═════════════╪═══════╡
│ 0   ┆ 6192     ┆ 2023-09-01  ┆ 2023-09-01  ┆ … ┆ LIT         ┆ NEW         ┆ XASE        ┆ XASE  │
│     ┆          ┆ 09:33:02.87 ┆ 13:33:02.87 ┆   ┆             ┆             ┆             ┆       │
│     ┆          ┆ 3930097     ┆ 3930097     ┆   ┆             ┆             ┆             ┆       │
│ 1   ┆ 34803    ┆ 2023-09-01  ┆ 2023-09-01  ┆ … ┆ LIT         ┆ NEW        